# WhisperMini: Urdu Whisper Fine-Tuning Notebook

This notebook fine-tunes OpenAI Whisper for Urdu ASR on your local dataset and saves the resulting model into the `models/` directory so it can be used by `backend/stt`.

**Pipeline overview**

- **Configure** paths and hyperparameters
- **Load** train/eval manifests with local Urdu audio + transcripts
- **Build** Hugging Face `datasets` objects
- **Load** `openai/whisper-tiny` + `WhisperProcessor` for Urdu transcription
- **Preprocess** data and create data collator
- **Fine-tune** with `Seq2SeqTrainer`
- **Evaluate** WER/CER on the eval set
- **Save** fine-tuned model for use in ORICALO


## 1. Environment Setup

Run this cell once per environment to install the required packages (if they are not already installed). You can skip it if your environment already has these libraries.

In [1]:
# Optional: install dependencies (uncomment if needed)
# !pip install -U "transformers[torch]" datasets accelerate jiwer evaluate librosa optimum[onnxruntime] torchaudio

import os
from pathlib import Path

import torch

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Torch version: 2.9.1+cpu
CUDA available: False


## 2. Project Paths and Configuration

Configure where your data and models live. Adjust these paths to match your local setup.

This notebook assumes it is located at `training/asr/whisper_urdu_finetune_full.ipynb` inside the ORICALO repo.

In [2]:
# 2.1 Paths and Hyperparameters

from pathlib import Path

# NOTE: Adjust PROJECT_ROOT if this does not resolve to your ORICALO repo root.
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = (NOTEBOOK_DIR / ".." / "..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "asr" / "urdu"
MODELS_DIR = PROJECT_ROOT / "models"

TRAIN_MANIFEST = DATA_DIR / "train_manifest.json"
EVAL_MANIFEST = DATA_DIR / "eval_manifest.json"

# Base Whisper checkpoint to fine-tune
BASE_MODEL_ID = "openai/whisper-tiny"  # change to small/base/medium if needed
LANGUAGE = "Urdu"
TASK = "transcribe"

# Where the fine-tuned model will be saved
OUTPUT_DIR = MODELS_DIR / "whisper-tiny-urdu-finetuned"

# Training hyperparameters (tune for your GPU and dataset size)
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
NUM_TRAIN_EPOCHS = 5
LEARNING_RATE = 1e-5
WARMUP_STEPS = 500
LOGGING_STEPS = 50
EVAL_STEPS = 500
SAVE_STEPS = 500
MAX_DURATION_SECONDS = 30.0  # filter very long audios for stability

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Notebook dir:", NOTEBOOK_DIR)
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Models dir:", MODELS_DIR)
print("Using device:", DEVICE)

Notebook dir: G:\Other computers\My Laptop\FAST-NUCES\S07-FALL-2025\FYP\ORICALO\training\asr
Project root: G:\Other computers\My Laptop\FAST-NUCES\S07-FALL-2025\FYP\ORICALO
Data dir: G:\Other computers\My Laptop\FAST-NUCES\S07-FALL-2025\FYP\ORICALO\data\asr\urdu
Models dir: G:\Other computers\My Laptop\FAST-NUCES\S07-FALL-2025\FYP\ORICALO\models
Using device: cpu


## 3. Dataset Manifests

This notebook expects **JSON manifests** for train and eval splits:

- `TRAIN_MANIFEST` (e.g. `data/asr/urdu/train_manifest.json`)
- `EVAL_MANIFEST`  (e.g. `data/asr/urdu/eval_manifest.json`)

Each file should contain a list of objects with at least:

- `audio_path`: path to a `.wav` file (absolute or relative to repo root)
- One of: `text`, `sentence`, or `reference_text`: ground-truth Urdu transcription

Example entry:

```json
{
  "audio_path": "data/asr/urdu/wavs/sample_0001.wav",
  "reference_text": "یہ ایک نمونہ جملہ ہے۔"
}
```

You can reuse the same manifest format as `backend/evaluation/baseline_evaluation.py` (`audio_path` + `reference_text`).

In [3]:
# 3.1 Load Manifests into Hugging Face Datasets

import json
from typing import List, Dict

import numpy as np
from datasets import Dataset, DatasetDict, Audio


def load_manifest_to_dataset(manifest_path: Path) -> Dataset:
    """Load a JSON or JSONL manifest into a `datasets.Dataset` with Audio + text.

    Supported keys:
    - audio path:  `audio_path` | `path` | `audio_filepath`
    - text field:  `text` | `sentence` | `reference_text`
    """
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest_path}")

    records: List[Dict] = []

    if manifest_path.suffix in {".jsonl", ".jsonl2"}:
        with open(manifest_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                records.append(json.loads(line))
    else:
        with open(manifest_path, "r", encoding="utf-8") as f:
            raw = json.load(f)
        if isinstance(raw, dict):
            raw = raw.get("data", [])
        records = list(raw)

    for rec in records:
        # Normalize text field
        if "text" not in rec:
            if "sentence" in rec:
                rec["text"] = rec["sentence"]
            elif "reference_text" in rec:
                rec["text"] = rec["reference_text"]
            else:
                raise KeyError(
                    "Manifest record missing text field; expected 'text', 'sentence', or 'reference_text'"
                )

        # Normalize audio path into an `audio` field compatible with datasets.Audio
        if "audio" not in rec:
            path = rec.get("audio_path") or rec.get("path") or rec.get("audio_filepath")
            if path is None:
                raise KeyError(
                    "Manifest record missing audio path; expected 'audio_path', 'path', or 'audio_filepath'"
                )
            rec["audio"] = {"path": str(path)}

    ds = Dataset.from_list(records)
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))

    # Keep only the fields we actually need
    keep_cols = {"audio", "text"}
    drop_cols = [c for c in ds.column_names if c not in keep_cols]
    if drop_cols:
        ds = ds.remove_columns(drop_cols)

    return ds


train_dataset = load_manifest_to_dataset(TRAIN_MANIFEST)
eval_dataset = load_manifest_to_dataset(EVAL_MANIFEST)

raw_datasets = DatasetDict({"train": train_dataset, "eval": eval_dataset})

print(raw_datasets)
print("Example train sample:")
print(raw_datasets["train"][0])

ModuleNotFoundError: No module named 'datasets'

## 4. Load Whisper Model and Processor (Urdu)

We use Hugging Face `WhisperProcessor` + `WhisperForConditionalGeneration`:

- Base checkpoint: `BASE_MODEL_ID` (e.g. `openai/whisper-tiny`)
- Language set to **Urdu**
- Task set to **transcribe** (no translation)

The processor handles both feature extraction (audio → log-mel) and tokenization.

In [ ]:
# 4.1 Initialize Processor and Model

from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    BASE_MODEL_ID,
    language=LANGUAGE,
    task=TASK,
)

model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_ID)

# Recommended for training (disable cache, allow full generation)
model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

model.to(DEVICE)

print("Loaded base model:", BASE_MODEL_ID)
print("Model device:", next(model.parameters()).device)

## 5. Preprocessing and Feature Extraction

We now:

- Optionally **filter** examples longer than `MAX_DURATION_SECONDS`.
- Convert raw audio into Whisper `input_features`.
- Tokenize Urdu transcripts into label `input_ids`.

All of this is stored in a processed `DatasetDict` ready for the Trainer.

In [ ]:
# 5.1 Filter Long Audio and Prepare Features

from typing import Any


def is_within_max_duration(batch: Dict[str, Any]) -> bool:
    if MAX_DURATION_SECONDS is None:
        return True
    audio = batch["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    return duration <= MAX_DURATION_SECONDS


def prepare_dataset(batch: Dict[str, Any]) -> Dict[str, Any]:
    # Audio → log-mel input features
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # Text → token IDs (labels)
    with processor.as_target_processor():
        batch["labels"] = processor(batch["text"]).input_ids

    return batch


# Filter by duration first
if MAX_DURATION_SECONDS is not None:
    raw_datasets["train"] = raw_datasets["train"].filter(is_within_max_duration)
    raw_datasets["eval"] = raw_datasets["eval"].filter(is_within_max_duration)

# Map to input_features + labels
processed_datasets = raw_datasets.map(
    prepare_dataset,
    remove_columns=raw_datasets["train"].column_names,
    num_proc=4,
)

print(processed_datasets)

## 6. Data Collator and Metrics (WER/CER)

We define:

- A **data collator** that pads input features and labels appropriately.
- A `compute_metrics` function returning **WER** and **CER** using `evaluate`.

This mirrors the evaluation you already have in `backend/evaluation/baseline_evaluation.py`.

In [ ]:
# 6.1 Data Collator and Metrics

from dataclasses import dataclass
from typing import List, Union

import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        # 1. Pad input_features
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # 2. Pad labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1), -100
        )

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


def compute_metrics(pred):
    """Compute WER and CER on generated predictions."""
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 in labels as we can't decode them
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer_value = wer_metric.compute(predictions=pred_str, references=label_str)
    cer_value = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer_value, "cer": cer_value}

## 7. Fine-Tuning with `Seq2SeqTrainer`

We now configure `Seq2SeqTrainingArguments` and launch fine-tuning:

- Mixed precision (`fp16`) if CUDA is available
- Evaluation + checkpointing every `EVAL_STEPS` / `SAVE_STEPS`
- Metrics: WER and CER

Depending on dataset size and GPU, training can take a while.

In [ ]:
# 7.1 Configure Trainer and Start Training

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=LOGGING_STEPS,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    predict_with_generate=True,
    generation_max_length=225,
    report_to=[],  # disable default logging integrations (e.g. wandb)
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["eval"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

trainer.train()

## 8. Evaluate and Save Fine-Tuned Model

After training, we:

- Run evaluation on the eval split (WER/CER).
- Save the fine-tuned model and processor into `models/whisper-tiny-urdu-finetuned`.

You can later upload this directory to Hugging Face Hub or point your backend to it as a local model.

In [ ]:
# 8.1 Evaluate on eval set

metrics = trainer.evaluate()
print("Eval metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

# 8.2 Save fine-tuned model + processor
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(OUTPUT_DIR))  # saves model + config
processor.save_pretrained(str(OUTPUT_DIR))

print(f"\n✅ Saved fine-tuned model and processor to: {OUTPUT_DIR}")

## 9. Optional: Export to ONNX and Dynamic Quantization

If you want to reproduce the ONNX + quantization results from your report, you can:

1. Export the fine-tuned model to ONNX using `optimum.onnxruntime`.
2. Apply **dynamic quantization** for CPU-friendly inference.

This section is optional and only needed if you plan to serve the ONNX model directly (outside the HF `pipeline`).

In [ ]:
# 9.1 Export fine-tuned model to ONNX (optional)

try:
    from optimum.onnxruntime import ORTModelForSpeechSeq2Seq

    ONNX_DIR = OUTPUT_DIR / "onnx"
    ONNX_DIR.mkdir(parents=True, exist_ok=True)

    ort_model = ORTModelForSpeechSeq2Seq.from_pretrained(
        OUTPUT_DIR,
        export=True,
        from_transformers=True,
    )
    ort_model.save_pretrained(ONNX_DIR)

    print(f"✅ Exported ONNX model to: {ONNX_DIR}")
except ImportError as e:
    print("⚠️ optimum.onnxruntime not installed. Install with:")
    print("   pip install 'optimum[onnxruntime]'\n", e)

In [ ]:
# 9.2 Dynamic quantization of ONNX model (optional)

try:
    from optimum.onnxruntime import ORTQuantizer
    from optimum.onnxruntime.configuration import AutoQuantizationConfig

    ONNX_DIR = OUTPUT_DIR / "onnx"
    QUANT_DIR = ONNX_DIR / "dynamic_quant"
    QUANT_DIR.mkdir(parents=True, exist_ok=True)

    # Use AVX2 dynamic quantization config (works on modern x86_64 CPUs, including Ryzen)
    qconfig = AutoQuantizationConfig.avx2(is_static=False)

    quantizer = ORTQuantizer.from_pretrained(ONNX_DIR)
    quantizer.quantize(save_dir=QUANT_DIR, quantization_config=qconfig)

    print(f"✅ Saved dynamically quantized ONNX model to: {QUANT_DIR}")
except ImportError as e:
    print("⚠️ optimum.onnxruntime not installed. Install with:")
    print("   pip install 'optimum[onnxruntime]'\n", e)
except Exception as e:
    print("⚠️ Quantization failed:", e)

## 10. Using the Fine-Tuned Model in `backend/stt`

Your current `backend/stt/stt_hf.py` uses:

- `Ear_hf(model_id="kingabzpro/whisper-tiny-urdu", device=...)`

To use the fine-tuned model from this notebook instead:

- **Option 1 (no code change):** When instantiating `Ear_hf`, pass the local path:
  - `model_id="models/whisper-tiny-urdu-finetuned"`

- **Option 2 (change default):** Update the default `model_id` in `Ear_hf.__init__` to:
  - `model_id: str = "models/whisper-tiny-urdu-finetuned"`

Because we saved with `WhisperForConditionalGeneration` + `WhisperProcessor`, the Hugging Face `pipeline` in `Ear_hf` can load this directory directly.

You can now:

- Fine-tune using this notebook.
- Evaluate with `backend/evaluation/baseline_evaluation.py` by passing `--model custom --model-id models/whisper-tiny-urdu-finetuned`.
- Compare baseline vs. fine-tuned WER/CER, as described in your report.